<a href="https://colab.research.google.com/github/kchenTTP/python-series/blob/main/web_apis/Consuming_Web_APIs_With_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Consuming Web APIs with Python**


## **Accessing this handout**

**Link:** [on.nypl.org/python](https://on.nypl.org/python)

To access this handout, first sign into Google, then select "Save a Copy in Drive" to make your own editable copy of this handout.


An **API** *(Application Programming Interface)* is the digital interface that allows different software applications to communicate and exchange data. In the context of the web, an API is a set of rules and protocols that define how a **client** *(a browser, a Python script, etc.)* can request information from a **server** *(the remote database, service, etc.)*.

In this session, we will cover the essentials of **networking** (how data travels across the internet) and practice interacting with different data sources using Python's requests library.

<br>

### **Table of Contents**
- [Networking 101](#scrollTo=EagKIm3iZL3Z)
- [Accessing APIs](#scrollTo=rjRdjirqh2nh)
- [Examples](#scrollTo=THtU81R02zSg)


## **Networking 101**

Information is exchanged on the web via the **hypertext transfer protocol (HTTP)**. Every time you use a browser or write code to fetch data, you are participating in an **HTTP Request & Response cycle**.


### **The Request & Response Cycle**

An API call is simply a structured exchange between a **Client** and a **Server**.

<br>

<figure align="center">
  <img src="https://raw.githubusercontent.com/kchenTTP/python-series/refs/heads/main/web_apis/assets/request_response_cycle.png" alt="The request and response cycle diagram" />
  <figcaption>Simplified request and response cycle</figcaption>
</figure>


#### **The Request _(What you send)_**

To initiate a connection, your script sends a request containing:

- **URL _(Endpoint)_:** The web address you are hitting

- **Parameters:** Extra information added to the URL _(ex: `?api_key=DEMO_KEY&date=2023-01-01`)_

- **[Method](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Methods):** The action you want to perform

  - `GET`: Retrieve data from a server

  - `POST`: Send or create new data on a server

- **[Headers](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Headers):** Metadata about the request _(we won't talk about headers in this class but every HTTP request and response has a header)_


#### **The Response _(What you get back)_**

After processing your request, the server sends back a response:

- **[Status Code](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Status):** A 3 digit number indicating the result

  - `200`: Success _(OK)_

  - `401`: Unauthorized _(Missing or invalid API Key)_

  - `404`: Not Found _(The URL doesn't exist)_

- **Body _(Payload)_:** The actual data, usually formatted as **JSON** _(JavaScript Object Notation)_


## **Accessing APIs**

While every service is slightly different, using most modern web APIs follow a standard 3 step workflow:

<br>

### 1. Read Documentation

The **API Documentation** is the instruction manual provided by the developers. It tells you:

- The purpose of the API

- The **Base URL** _(the root address of the API)_

- The **Endpoints** _(the paths for different resources)_

- The **Required Parameters** _(what info you need to provide)_

### 2. Secure Authentication

Most APIs require you to sign up for an account to receive an **API Key**, which acts as a digital *"passcard"* to identify your requests. While there are other ways of authentication, we'll be using APIs that use an API key for authentication or ones that don't require one.

> ❗ **IMPORTANT**: **DO NOT** share your authentication info publicly or save it directly in your code. If someone steals your key, they can use up your credits, get your account banned, or worse.

### 3. Confirm Rate Limits

APIs often have **Rate Limits**, which is a cap on how many requests you can make per a set period of time. Going over the limit will usually result in a **429 Too Many Requests** error.


## **Import Libraries**

We'll be using the following libraries for this session:

- [`requests`](https://docs.python-requests.org/en/latest/index.html): Handles simple HTTP requests and responses in Python.

- [`google.colab`](https://github.com/googlecolab/colabtools): Provides utilities to securely access environment variables within the Colab environment.

- [`pprint`](https://docs.python.org/3/library/pprint.html): Short for "pretty-print"; it formats complex data structures into more readable layouts.

- [`IPython.display`](https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html#module-IPython.display): Used to render rich content like images, HTML, or videos directly inside your notebook cells.


In [ ]:
import requests
from google.colab import userdata
from pprint import pp
from IPython.display import display, Image

## **Example: OpenWeather API**

OpenWeather is an online service that provides real-time, historical, and forecasted weather data. In this example, we will be using the **Current Weather Data** endpoint and **5 Day Forecast** endpoint to retrieve atmospheric conditions for a specific city.


### **Preparation**

- **API Key:** Sign up at the [OpenWeather Portal](https://home.openweathermap.org/users/sign_up) to receive your API key _(`appid`)_.

- **Check Quotas:** Review the [Pricing Page](https://openweathermap.org/price#:~:text=Free%20for%20everyone-,Current,-weather) to understand request limits.


### **Storing API Keys**

> ❗ **DO NOT** store your API keys directly in your code.  
> 💡 You should instead store API keys as **Environment Variables!**

<br>

Google Colab allows you to store secrets _(api keys, passwords, sensative information, ...)_ as environment variables:


1. Click on the `Secrets` menu _(key icon)_ on the left side to manage your environment variables.

2. Click on `+ Add new secret`.

3. Type `WEATHER_KEY` into the "Name" field and paste your newly obtained API key into the "Value" field _(cannot contain space)_.

4. Turn on "Notebook access"


In [ ]:
# Get environment variables from colab like this, make sure you use the exact same name

WEATHER_KEY = userdata.get('WEATHER_KEY')

> 💡 **NOTE**: For environment variables and constants, `SCREAMING_SNAKE_CASE` is the convention


### **Get Current Weather Data For Location**

To get current weather data, we first need to convert a location name or zip code into geographic coordinates _(**Latitude** and **Longitude**)_. We do this using the **Geocoding API**.


#### **Step 1: Get Coordinates _(Geocoding)_**

- **Endpoint:** `http://api.openweathermap.org/geo/1.0/zip?zip={zip code},{country code}&appid={API key}`

- **Documentation:** [Geocoding API](https://openweathermap.org/api/geocoding-api)


In [ ]:
# Store the API endpoint
geo_endpoint = "http://api.openweathermap.org/geo/1.0/zip"

In [ ]:
# Store the parameters in a dictionary to pass into the requests.get method

params = {
  "appid": WEATHER_KEY,
  "zip": 10016
}

In [ ]:
# Pass the API endpoint and parameters into the requests.get method
response = requests.get(geo_endpoint, params=params)

# Raises an error if response status is not 200
response.raise_for_status()

# Convert JSON data to Python dictionary
loc_info = response.json()

pp(loc_info)

#### **Step 2: Get Weather _(Current Weather)_**

Once we have the coordinates, we pass them into the **Current Weather Data** endpoint to retrieve live atmospheric conditions.

- **Endpoint:** `https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={API key}`

- **Documentation:** [Current Weather Data](https://openweathermap.org/current)

Read the documentation to identify which **parameters** (like `units` or `lang`) you can use to modify results.


In [ ]:
# Store the API endpoint
weather_endpoint = "https://api.openweathermap.org/data/2.5/weather"

In [ ]:
# Your code goes here

# Step 1: Create a dictionary that holds all the parameters (including your API key)
params = {}

# Step 2: Make the api call and store the response in a variable named `current_weather`


In [ ]:
#@title Solution

params = {
  "appid": WEATHER_KEY,
  "lat": loc_info["lat"],
  "lon": loc_info["lon"],
  "units": "imperial"
}

response = requests.get(weather_endpoint, params=params)

response.raise_for_status()

current_weather = response.json()

pp(current_weather)

### **Get Weather Forecast For Location**

To get the weather forecast of a location, call the **5 Day Forecast** endpoint.

- **Endpoint:** `https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API key}`

- **Documentation:** [5 Day Forecast](https://openweathermap.org/forecast5?collection=current_forecast)


In [ ]:
# Store the API endpoint
forecast_endpoint = "https://api.openweathermap.org/data/2.5/forecast"

In [ ]:
# Your code goes here

# Step 1: Create a dictionary that holds all the parameters (including your API key)
params = {}

# Step 2: Make the api call and store the response in a variable named `weather_forecast`


In [ ]:
#@title Solution

params = {
  "appid": WEATHER_KEY,
  "lat": loc_info["lat"],
  "lon": loc_info["lon"],
  "units": "imperial",
  "cnt": 5
}

response = requests.get(forecast_endpoint, params=params)

response.raise_for_status()

weather_forecast = response.json()

pp(weather_forecast)

### **Parsing the Information**

Now that we've got the full JSON response for our weather forecast, let's extract some details to create a summary for our weather forecast.


In [ ]:
# Store the name of the city
city = weather_forecast.get("city").get("name")
pp(city)

In [ ]:
# Store the information we need from the list of forecast

forecast_list = []

for timestamp in weather_forecast.get("list"):
  forecast_list.append({
      "datetime": timestamp.get("dt_txt"),
      "weather": timestamp.get("weather")[0].get("main"),
      "temperature": timestamp.get("main").get("temp"),
      "humidity": timestamp.get("main").get("humidity")
  })

pp(forecast_list)

### **Weather Maps**

The OpenWeather API provides various weather map layers. To view them, we use a specific endpoint that requires **Z** (Zoom), **X**, and **Y** coordinates.

- **Endpoint:** `https://tile.openweathermap.org/map/{layer}/{z}/{x}/{y}.png?appid={API key}`
- **Documentation:** [Weather Maps 1.0](https://openweathermap.org/api/weathermaps?collection=maps)

> **NOTE**: The `x` & `y` coordinates are [**Slippy Map tilenames**](https://wiki.openstreetmap.org/wiki/Slippy_map_tilenames). We must convert standard Latitude and Longitude into these tile numbers using the functions below.


In [ ]:
#@title Conversion function for x, y coordinates
import math

def deg2tile(lat_deg, lon_deg, zoom):
  """Converts GPS coordinates to OSM/OpenWeather tile (x, y)."""
  lat_rad = math.radians(lat_deg)
  n = 1 << zoom
  xtile = int((lon_deg + 180.0) / 360.0 * n)
  ytile = int((1.0 - math.asinh(math.tan(lat_rad)) / math.pi) / 2.0 * n)
  return xtile, ytile

def tile2deg(xtile, ytile, zoom):
  """Converts OSM/OpenWeather tile (x, y) to GPS coordinates."""
  n = 1 << zoom
  lon_deg = xtile / n * 360.0 - 180.0
  lat_rad = math.atan(math.sinh(math.pi * (1 - 2 * ytile / n)))
  lat_deg = math.degrees(lat_rad)
  return lat_deg, lon_deg

#### **Displaying a Raw Weather Tile**

If we request a single tile and display it directly, it looks like a floating square of color because it lacks the actual map showing all the context (street, borders, etc.).


In [ ]:
layer = "temp_new"
z = 0  # Zoom level 0: world wide
x, y = deg2tile(loc_info["lat"], loc_info["lon"], z)

map_endpoint = f"https://tile.openweathermap.org/map/{layer}/{z}/{x}/{y}.png"
response = requests.get(map_endpoint, params={"appid": WEATHER_KEY})
response.raise_for_status()

display(Image(response.content))

#### **Interactive Weather Overlay with Folium**

To see the weather layer on a real map, we are going to use [`folium`](https://python-visualization.github.io/folium/latest/getting_started.html). It automatically calculates the tiles needed and overlays them onto OpenStreetMap.


In [ ]:
import folium

layer = "temp_new"
location = [loc_info["lat"], loc_info["lon"]]
m = folium.Map(location=location, zoom_start=5)

# Define the weather tile URL
weather_url = (
    f"https://tile.openweathermap.org/map/{layer}/"
    "{z}/{x}/{y}.png"
    f"?appid={WEATHER_KEY}"
)

# Add the weather overlay
folium.TileLayer(
    tiles=weather_url,
    attr='OpenWeather',
    name='Temperature',
    overlay=True,
    opacity=0.7
).add_to(m)

# Add the layer toggle
folium.LayerControl().add_to(m)

m

Try out other map layers too!

- **Clouds**: `clouds_new`
- **Precipitation**: `precipitation_new`
- **Sea level pressure**: `pressure_new`
- **Wind speed**: `wind_new`
- **Temperature**: `temp_new`


Congrats! You've successfully mastered the basics of consuming a web API. OpenWeather offers a few more free endpoints to view **air pollution** and **weather maps**.

Check out the full [OpenWeather Documentation](https://openweathermap.org/api/air-pollution?collection=environmental) to discover more.


## **Conclusion**

We've covered how to fetch, parse, and display complex data from different sources on the web. If you'd like to dive deeper into the fundamentals of networking that make these API calls possible, check out these resources:

- [HTTP Fundamentals - GeeksForGeeks](https://www.geeksforgeeks.org/computer-networks/http-tutorial/)
- [OSI Model (The 7 Layers of The Internet) - Wikipedia](https://en.wikipedia.org/wiki/OSI_model)

To discover more APIs for your next project, here are some links:

- [Public APIs List - GitHub](https://github.com/public-apis/public-apis)

We also offer a wide variety of Python classes which you can find: [**here**](https://www.nypl.org/techconnect?keyword=python)
